# DuckPD Advanced Features & New Capabilities Walkthrough

Welcome to the **DuckPD Features Walkthrough**! This notebook demonstrates the newest capabilities of DuckPD on **real-world financial market data** using the [AlphaDojo/dojo_stock_news](https://huggingface.co/datasets/AlphaDojo/dojo_stock_news) dataset (~3.9M articles).

### What you will see:
- **Direct Remote Parquet Scanning**: Query millions of rows in cloud parquet without loading full datasets into Python memory.
- **Vectorized String Accessors (`.str`)**: Clean publisher names, extract headlines, and filter topics lazily.
- **Multi-Table Relational Merges (`merge`)**: Join multi-million row news feeds with ticker metadata tables.
- **Multi-Frame Concatenation (`duckpd.concat`)**: Combine filtered partitions with automatic schema union and null-padding.
- **Extended Reductions**: Compute standard deviation (`std`), variance (`var`), median (`median`), and quantiles (`quantile`).
- **Advanced Multi-Column GroupBy**: Named aggregations across publishers and tickers.
- **Window & Positional Transforms**: Cumulative, rank, difference, rolling, expanding, and shifted analytics with guaranteed ordering.
- **Persistence, Query Plans & Direct Parquet Export**: Reuse materialized intermediates, inspect pushdown, and write without pandas fallback.

## 1. Setup Session & Connect to Remote Parquet

Initialize a DuckPD session with custom memory and execution settings, then lazily scan the 3.9M row dataset hosted on Hugging Face.

In [1]:
import pandas as std_pd

import duckpd as pd

print(f"DuckPD Version: {pd.__version__}")
session = pd.connect(memory_limit="1GB", threads=4)

# Remote dataset from AlphaDojo (~3.9M financial news rows)
DATA_URL = "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"

# Lazily scan parquet directly over HTTP without downloading whole file into memory
news_df = session.read_parquet(DATA_URL)

print("Lazy DataFrame created:")
print(f"Columns: {news_df.columns}")
print(f"Session executions so far: {session.execution_count}")

DuckPD Version: 0.0.7
Lazy DataFrame created:
Columns: ('title', 'image', 'ago', 'primarysymbol', 'primarytopic', 'publisher', 'url', 'id', 'imagedomain', 'description', 'primarytopic_url', 'publisher_logo', 'publish_date', 'on_symbol_json', 'symbol', 'source')
Session executions so far: 0


## 2. Vectorized String Accessors (`.str`)

Clean publisher names, compute headline lengths, and flag earnings-related announcements lazily using DuckPD's `.str` accessor methods.

In [2]:
# Perform lazy string feature engineering
enriched_news = news_df.assign(
    publisher_clean=news_df["publisher"].str.strip().str.upper(),
    title_len=news_df["title"].str.len(),
    is_earnings=news_df["title"].str.upper().str.contains("EARNINGS"),
    is_option_activity=news_df["title"].str.contains("Option Activity"),
)

# Inspect a bounded preview pushed down to DuckDB
enriched_news[["symbol", "publisher_clean", "title_len", "is_earnings", "title"]].head(
    5
)

,symbol,publisher_clean,title_len,is_earnings,title
0,TWOX,BNK INVEST,88,False,Friday Sector Laggards: Education & Training S...
1,PBTP,BNK INVEST,92,False,PIMCO California Municipal Income Fund Breaks ...
2,PBTP,ZACKS,65,False,SYK Stock Gains 3.8% Since March-End: What's D...
3,PBTP,THE MOTLEY FOOL,100,False,A Once-in-a-Decade Opportunity: 1 Super Growth...
4,PBTP,ZACKS,60,False,Can BMY's First CELMoD Approval Support Its Gr...


## 3. Multi-Table Relational Merging (`merge`)

Join the multi-million row news dataset with a ticker reference metadata table. The join and predicates are compiled into relational SQL execution.

In [3]:
# Reference table for prominent tech & consumer market cap leaders
ticker_meta = session.from_pandas(
    std_pd.DataFrame(
        {
            "symbol": ["AAPL", "NVDA", "MSFT", "AMZN", "TSLA", "GOOGL"],
            "company_name": [
                "Apple Inc.",
                "NVIDIA Corp.",
                "Microsoft Corp.",
                "Amazon.com Inc.",
                "Tesla Inc.",
                "Alphabet Inc.",
            ],
            "sector": [
                "Technology",
                "Semiconductors",
                "Software",
                "E-Commerce",
                "Automotive",
                "Communication",
            ],
            "market_tier": [
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
            ],
        }
    )
)

# Merge ticker metadata with news stream
news_with_sector = ticker_meta.merge(enriched_news, on="symbol", how="inner")

news_with_sector[
    ["symbol", "company_name", "sector", "publisher_clean", "title_len", "title"]
].head(5)

,symbol,company_name,sector,publisher_clean,title_len,title
0,NVDA,NVIDIA Corp.,Semiconductors,ZACKS,67,The AI Stock Everyone Knows - But Few See Beco...
1,NVDA,NVIDIA Corp.,Semiconductors,ZACKS,52,The Optical Shift: Why AAOI is Built for the A...
2,NVDA,NVIDIA Corp.,Semiconductors,THE MOTLEY FOOL,53,I Think You Missed CoreWeave's Zero-Cost-Basis...
3,NVDA,NVIDIA Corp.,Semiconductors,ZACKS,63,The Zacks Analyst Blog Highlights Seagate Tech...
4,NVDA,NVIDIA Corp.,Semiconductors,ZACKS,61,Utility ETFs to Buy as Rapid AI Buildout Spark...


## 4. Multi-Frame Concatenation (`duckpd.concat`)

Combine distinct ticker news subsets row-wise with automatic schema union and null-padding.

In [4]:
# Split subsets and enrich one partition with custom category tags
nvda_news = news_with_sector[news_with_sector["symbol"] == "NVDA"].assign(
    focus_area="AI Hardware"
)[["symbol", "company_name", "focus_area", "publisher_clean", "title"]]

tsla_news = news_with_sector[news_with_sector["symbol"] == "TSLA"][
    ["symbol", "company_name", "publisher_clean", "title"]
]

# Concatenate partitions: focus_area will be padded with NULLs for TSLA
combined_stream = pd.concat([nvda_news, tsla_news])
print("Union Columns:", combined_stream.columns)

combined_stream.head(6)

Union Columns: ('symbol', 'company_name', 'focus_area', 'publisher_clean', 'title')


,symbol,company_name,focus_area,publisher_clean,title
0,NVDA,NVIDIA Corp.,AI Hardware,ZACKS,The AI Stock Everyone Knows - But Few See Beco...
1,NVDA,NVIDIA Corp.,AI Hardware,ZACKS,The Optical Shift: Why AAOI is Built for the A...
2,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,I Think You Missed CoreWeave's Zero-Cost-Basis...
3,NVDA,NVIDIA Corp.,AI Hardware,ZACKS,The Zacks Analyst Blog Highlights Seagate Tech...
4,NVDA,NVIDIA Corp.,AI Hardware,ZACKS,Utility ETFs to Buy as Rapid AI Buildout Spark...
5,NVDA,NVIDIA Corp.,AI Hardware,ZACKS,Top Big Data Stocks Tapping the Surging Demand...


## 5. Extended Statistical & Boolean Reductions

Calculate statistical metrics across headline length and content properties (`mean`, `median`, `std`, `var`, `quantile`, `any`, `all`) computed in a single SQL query in DuckDB.

In [5]:
print("--- Headline Length Statistical Metrics ---")
print(f"Mean Length:       {news_with_sector['title_len'].mean():.2f}")
print(f"Median Length:     {news_with_sector['title_len'].median():.2f}")
print(f"Std Deviation:     {news_with_sector['title_len'].std():.2f}")
print(f"Variance:          {news_with_sector['title_len'].var():.2f}")
print(f"25th Percentile:   {news_with_sector['title_len'].quantile(0.25):.2f}")
print(f"75th Percentile:   {news_with_sector['title_len'].quantile(0.75):.2f}")
print(f"95th Percentile:   {news_with_sector['title_len'].quantile(0.95):.2f}")

print("\n--- Boolean Reductions on Filtered Subset ---")
print(f"All headlines mention earnings? {news_with_sector['is_earnings'].all()}")
print(f"Any headline mentions earnings? {news_with_sector['is_earnings'].any()}")

--- Headline Length Statistical Metrics ---
Mean Length:       75.82
Median Length:     69.00
Std Deviation:     28.06
Variance:          787.22
25th Percentile:   57.00
75th Percentile:   89.00
95th Percentile:   130.00

--- Boolean Reductions on Filtered Subset ---
All headlines mention earnings? False
Any headline mentions earnings? True


## 6. Advanced GroupBy & Multi-Metric Aggregations

Perform analytical grouping across publishers and tickers using named aggregations, calculating article volume, average length, dispersion, and extreme values.

In [6]:
# Aggregate news analytics by publisher across top market-cap tickers
publisher_analytics = (
    news_with_sector.groupby(["publisher_clean"], as_index=False)
    .agg(
        article_count=("title", "count"),
        avg_headline_len=("title_len", "mean"),
        std_headline_len=("title_len", "std"),
        max_headline_len=("title_len", "max"),
        min_headline_len=("title_len", "min"),
    )
    .sort_values("article_count", ascending=False)
)

publisher_analytics.head(10)

,publisher_clean,article_count,avg_headline_len,std_headline_len,max_headline_len,min_headline_len
0,THE MOTLEY FOOL,729,84.652949,30.755979,195,25
1,ZACKS,297,63.353535,13.127478,112,29
2,MARKETBEAT,55,60.563636,11.370468,83,33
3,BARCHART,51,57.098039,10.696270,85,37
4,BNK INVEST,36,46.361111,17.568076,115,23
5,RTTNEWS,16,68.062500,20.638051,95,29
6,NASDAQ.COM,15,92.800000,16.699872,113,74
7,KEVIN DAVITT,1,62.000000,NaN,62,62


## 7. Window & Positional Transforms (`cumsum`, `rank`, `diff`)

Execute analytical window operations over ordered streams. DuckPD ensures window operations execute cleanly in DuckDB without in-memory materialization and validates that explicit `order_by` or `sort_values` guarantees are present.

In [7]:
# Compute cumulative article counts, volume ranks, and incremental step differences
ranked_publishers = publisher_analytics.assign(
    volume_rank=publisher_analytics["article_count"].rank(
        method="dense", ascending=False
    ),
    cumulative_articles=publisher_analytics["article_count"].cumsum(),
    article_step_diff=publisher_analytics["article_count"].diff(-1),
)

ranked_publishers.head(10)

,publisher_clean,article_count,avg_headline_len,std_headline_len,max_headline_len,min_headline_len,volume_rank,cumulative_articles,article_step_diff
0,THE MOTLEY FOOL,729,84.652949,30.755979,195,25,1.0,729,432.0
1,ZACKS,297,63.353535,13.127478,112,29,2.0,1026,242.0
2,MARKETBEAT,55,60.563636,11.370468,83,33,3.0,1081,4.0
3,BARCHART,51,57.098039,10.696270,85,37,4.0,1132,15.0
4,BNK INVEST,36,46.361111,17.568076,115,23,5.0,1168,20.0
5,RTTNEWS,16,68.062500,20.638051,95,29,6.0,1184,1.0
6,NASDAQ.COM,15,92.800000,16.699872,113,74,7.0,1199,14.0
7,KEVIN DAVITT,1,62.000000,NaN,62,62,8.0,1200,NaN


## 8. Rolling, Expanding & Shifted Analytics

Build richer ordered analytics with row-based rolling and expanding windows. The transforms stay lazy and compile into DuckDB window expressions; persisting then creates a reusable DuckDB table at an explicit execution boundary.

In [8]:
publisher_windows = ranked_publishers.assign(
    rolling_3_avg_articles=ranked_publishers["article_count"]
    .rolling(3, min_periods=1)
    .mean(),
    expanding_articles=ranked_publishers["article_count"].expanding().sum(),
    previous_publisher_articles=ranked_publishers["article_count"].shift(1),
)

persisted_publishers = publisher_windows.persist("publisher_window_summary")
print(f"Executions after persist: {session.execution_count}")
persisted_publishers.head(10)

Executions after persist: 15


,publisher_clean,article_count,avg_headline_len,std_headline_len,max_headline_len,min_headline_len,volume_rank,cumulative_articles,article_step_diff,rolling_3_avg_articles,expanding_articles,previous_publisher_articles
0,THE MOTLEY FOOL,729,84.652949,30.755979,195,25,1.0,729,432.0,729.000000,729.0,NaN
1,ZACKS,297,63.353535,13.127478,112,29,2.0,1026,242.0,513.000000,1026.0,729.0
2,MARKETBEAT,55,60.563636,11.370468,83,33,3.0,1081,4.0,360.333333,1081.0,297.0
3,BARCHART,51,57.098039,10.696270,85,37,4.0,1132,15.0,134.333333,1132.0,55.0
4,BNK INVEST,36,46.361111,17.568076,115,23,5.0,1168,20.0,47.333333,1168.0,51.0
5,RTTNEWS,16,68.062500,20.638051,95,29,6.0,1184,1.0,34.333333,1184.0,36.0
6,NASDAQ.COM,15,92.800000,16.699872,113,74,7.0,1199,14.0,22.333333,1199.0,16.0
7,KEVIN DAVITT,1,62.000000,NaN,62,62,8.0,1200,NaN,10.666667,1200.0,15.0


## 9. Plan Inspection (`explain()`) & Direct Export

Inspect the relational query plan generated by DuckPD, including predicate pushdown, joins, and window functions. Then write the window-enriched summary directly to Parquet without routing the full result through pandas.

In [9]:
print("=== Compiled Query Plan with Window Transforms ===")
print(publisher_windows.explain())

# Export the analytical summary directly from DuckDB.
publisher_windows.write_parquet("stock_news_summary.parquet", overwrite=True)
print("\nExported stock_news_summary.parquet directly via DuckDB!")

=== Compiled Query Plan with Window Transforms ===


DuckPD logical plan:
ProjectPlan(input=ProjectPlan(input=ProjectPlan(input=ProjectPlan(input=ProjectPlan(input=ProjectPlan(input=SortPlan(input=AggregatePlan(input=JoinPlan(left=ScanPlan(source=PandasSource(key='3edf204b9b584d139a5ff80efeae485b'), metadata=FrameMetadata(columns=(Column(id=ColumnId(value=UUID('a15384d7-80a1-44c6-b0cb-58de504ca7f4')), label='symbol', duckdb_type='VARCHAR', hidden=False, row_identity=False), Column(id=ColumnId(value=UUID('d83bd833-f9c7-4b11-bc5e-f782d03b9d2b')), label='company_name', duckdb_type='VARCHAR', hidden=False, row_identity=False), Column(id=ColumnId(value=UUID('d2a6d6de-e61b-45d2-8109-c310c0f6ac35')), label='sector', duckdb_type='VARCHAR', hidden=False, row_identity=False), Column(id=ColumnId(value=UUID('4c9bf7ef-7a8c-41c7-9bce-3a874cf052f1')), label='market_tier', duckdb_type='VARCHAR', hidden=False, row_identity=False), Column(id=ColumnId(value=UUID('12660315-b164-4e86-a021-f83aaa95f9ee')), label='__duckpd_row_ordinal_3edf204b9b584d139a5ff80ef